- 计划性停止 Early Stopping
- 突发性中断处理

##### 基于指标的自动提前停止 Early Stopping
##### 处理外部中断信号（SIGINT/SIGTERM）
##### 利用深度学习框架的原生回调（Callbacks）
##### 分布式训练中的特殊考虑

### 基于指标的自动提前停止 Early Stopping

这是常见的场景：当验证集损失不再下降或准确率不再提升时，自动停止训练并保存表现好的那个模型，而不是最后一个模型

核心逻辑
- 每个epoch结束后评估验证集指标
- 如果指标优于历史最佳，更新best_metric 并立即保存checkpoint
- 如果指标在连续N个epoch内没有改善，则触发停止信号
- 在停止前，确保加载或保留那个best_metric对应的权重

In [ ]:
class EarlyStopping:
    def __init__(self,patience=7,delta=0,path='checkpoing_best.pt'):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.path = path
        self.val_loss_min = flot('inf')
        
    def __call__(self,val_loss,model):
        score = -val_loss # 假设我们需要最小化loss，所以取负值作为分数
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss,model)
        elif score < self.best_score + self.delta:
            self.counter +=1
            print(f"EarlyStopping counter:{self.counter} out of {self.patience})
            if self.counter >= self.patience:
                  self.early_stop = True
        else:
                  self.best_score = score
                  self.save_checkpoint(val_loss,model)
                  self.counter = 0
    def save_checkpoint(self,val_loss,model):
        '''
        当模型性能提升时，保存checkpoint
        '''
        print(f'Validation loss decreased({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model ...')
        torch.save(model.state_dict(),self.path)     
        self.val_loss_min = val_loss
    
# 在训练循环中使用
# patience = 5，5个epoch不提示就停止
early_stopping = EarlyStopping(patience = 5,path='best_model.pth')
epochs = 2
for epoch in range(epochs):
    train(...)    
    val_loss = validate(...)
    early_stopping(val_loss,model)
    if early_stopping.early_stop:
        print('early stopping trggeerd. training finished.')
        break
    # 可选：同时保存最近一个的checkpoint，以防万一
    torch.save({
        'epoch':epoch,
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
    },f'checkpoint_epoch_{epoch}.pt')

### 处理外部中断信号（SIGINT/SIGTERM）

在大模型训练中，用户可能会手动按下 Ctrl+c ,或者集群调度系统因为资源回收发送终止信号，如果不处理，直接退出会导致当前epoch的数据和装填丢失。

核心逻辑：
- 注册信号处理器，捕获SIGINT(键盘中断)或SIGTERM(终止请求)。一旦捕获，设置一个标志位，让当前迭代完成后优雅退出，并在退出前强制保存一个checkpoint。


<b>总结工作流程</b>
1. 程序开始训练。
2. 用户按下 Ctrl+C。
3. Python 暂停当前计算，执行 signal_handler。
4. signal_handler 打印消息，并将 stop_requested 设为 True，然后返回。
5. 程序回到训练循环。
6. 如果在 batch 循环中，它会完成当前 batch 的计算，然后遇到 if stop_requested: break 跳出 batch 循环。
7. 进入 epoch 循环末尾，再次检测到 stop_requested 为 True。
8. 执行 torch.save，将完整的训练状态保存到 interrupt_checkpoint.pt。
9. 执行 break 跳出 epoch 循环，程序正常结束

In [ ]:
import signal 
import sys
import torch
from torch.utils.data import DataLoader

stop_requested = False

# 信号处理
def signal_handler(sig,frame):
    '''
    它不直接退出程序，只是把 stop_requested 设为 True 并打印提示。
    这是一种“软中断”策略，把退出的决定权交还给主循环。
    '''
    # 声明要修改全局变量。如果不加这句，函数内会被视为创建了一个新的局部变量，外部的循环无法感知变化。
    global stop_requested
    print("\n Interrupt received! saving checkpoint before existing...")
    stop_requested = True
    
# 注册信号
signal.signal(signal.SIGINT,signal_handler)
signal.signal(signal.SIGTERM,signal_handler)

# 训练循环
for epoch in range(total_epochs):
    for batch_idx,(data,target) in enumerate(train_loader):
        '''
        收到信号 -> 标记 stop_requested = True -> 继续跑完当前的 batch -> 甚至跑完当前的 epoch -> 再保存。
        '''
        # 中断请求
        if stop_requested:
            print('stopping gracefully at the end of this epoch...')
            # 保存包含优化器状态的完整checkpoint，以便恢复训练
            torch.save({
                'epoch':epoch,
                'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer_state_dict(),
                'loss':current_loss,
            },'interrupt_checkpoint_epoch.pt')
            break
            
       # 正常训练流程
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        # 可选：在batch结束后也检查（更灵敏）
        if stop_requested:
            break
    


In [ ]:
import signal
import sys
import torch

stop_requested = False

def signal_handler(sig, frame):
    global stop_requested
    print("\nInterrupt received! Saving checkpoint before exiting...")
    stop_requested = True

# 注册信号
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

try:
    # 训练循环
    for epoch in range(total_epochs):
        if stop_requested:
            print("Stopping gracefully at the end of this epoch...")
            # 1. 保存断点
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': current_loss,
            }, 'interrupt_checkpoint.pt')
            
            # 2. 跳出循环 (不要在这里 sys.exit，让代码继续往下走)
            break
            
        # 正常的训练步骤
        for batch in dataloader:
            # ... 训练代码 ...
            if stop_requested:
                break 
        
        if stop_requested:
            break

    # ---------------------------------------------------------
    # 【最佳位置】循环结束后在这里处理退出
    # 循环结束后，程序自然运行到脚本末尾，自动退出。
    # ---------------------------------------------------------
    
    if stop_requested:
        print("Training stopped by user. Checkpoint saved.")
        # 这里可以添加其他清理工作，比如关闭 writer, 释放资源等
        # writer.close() 
        
        # 显式退出，状态码 0 表示正常退出（虽然是中断，但我们成功保存了，算正常）
        sys.exit(0) 
    else:
        print("Training completed successfully!")
        # 正常训练结束也可以保存一个最终的 checkpoint
        torch.save({...}, 'final_checkpoint.pt')
        sys.exit(0)

except Exception as e:
    print(f"An unexpected error occurred: {e}")
    sys.exit(1) # 非 0 表示异常退出

### 在多进程训练（Distributed Data Parallel）中的处理

需要确保所有进程同步中断：

In [ ]:
import torch.distributed as dist

def should_stop(killer):
    """在分布式环境中同步停止信号"""
    stop_tensor = torch.tensor(1.0 if killer.kill_now else 0.0).cuda()
    dist.all_reduce(stop_tensor, op=dist.ReduceOp.MAX)
    return stop_tensor.item() > 0

# 在训练循环中
for batch in train_loader:
    if should_stop(killer):
        if dist.get_rank() == 0:
            save_checkpoint()
        dist.barrier()  # 等待所有进程
        cleanup()
        sys.exit(0)

####  如何恢复训练


In [ ]:
checkpoint = torch.load('interrupt_checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
start_batch = checkpoint.get('batch_idx', 0)

print(f'resuming from epoch {start_epoch}')

for epoch in range(start_epoch, num_epochs):
    # 使用sampler跳过已训练的样本（需要自定义Sampler）
    if epoch == start_epoch and start_batch > 0:
        # 或使用 itertools.islice
        from itertools import islice
        batch_iterator = islice(train_loader, start_batch, None)
    else:
        batch_iterator = train_loader
    
    for batch_idx, batch in enumerate(batch_iterator, start=start_batch if epoch == start_epoch else 0):
        train_step(batch)

## 利用深度学习框架的原生回调(Callbacks)

如果你使用huggingface transformers(Trainer)或pytroch Lighting,他们已经内置了完善的机制，无需手写上述逻辑

### huggingface Trainer
Tranier 原生支持 EarlyStoppingCallback和自动保存策略


In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="epoch",          # 每个 epoch 保存一次
    load_best_model_at_end=True,    # 训练结束后自动加载最佳模型
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # 其他参数...
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # 3个epoch不提升则停止
)

trainer.train()
# 训练结束后，output_dir 中会自动保留 best_model 和最后的 checkpoint

### pytorch Lightning
使用ModelCheckpoint 和EarlyStopping回调


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# 定义保存策略
checkpoint_callback = ModelCheckpoint(
    dirpath="./checkpoints",
    filename="{epoch}-{val_loss:.2f}",
    monitor="val_loss",
    mode="min",
    save_top_k=3, # 保存最好的3个
    save_last=True, # 始终保存上一个 epoch 的状态 (last.ckpt)
)

# 定义早停策略
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=5,
    verbose=True,
    mode="min"
)

trainer = Trainer(callbacks=[checkpoint_callback, early_stop_callback])
trainer.fit(model)

## 分布式训练中的特殊考虑

在大模型分布式训练(DDP,DeepSpeed,FSDP)中，停止和保存徐英浩特别注意<b>进程同步</b>
1. 主进程决策：通常右Rank 0 进程计算验证指标并决定是否停止
2. 广播信号： Rank 0 决定停止后，必须通过dist.broadcast或其他通信源语通知所有其他GPU进程，确保大家同时停止，避免死锁
3. 聚合保存
  - 如果是DeepSpeed或FSDP，模型权重被分片在不同GPU上，保存checkpoint时，需要调用框架特有的save_checkpoint方法，他会自动处理分片权重的收集（Gather）和写入磁盘，或者保存分片格式以便下次直接加载分片训练
  - 切记让每个进程都尝试写入同行一个文件，这会损坏数据

## 代码实现：在大模型任务中，每隔固定步数自动保存一次checkpoint + 开启early stopping 监控验证集 + 注册信号处理以应对突发中断。

核心设计思路
1. 定期保存 (Periodic Checkpoint): 每 save_every_n_steps 步保存一次，防止长时间训练后意外丢失。
2. 早停机制 (Early Stopping): 监控验证集 Loss，如果连续 patience 个 epoch 没有提升，则自动停止并保存最佳模型。
3. 信号处理 (Signal Handling): 捕获 Ctrl+C 或 SIGTERM，设置标志位，等待当前 Epoch 结束，保存“中断断点”，然后优雅退出。
4. 状态管理: 区分“最佳模型” (best_checkpoint.pt)、“定期备份” (backup_step_{}.pt) 和“中断恢复点” (interrupt_checkpoint.pt)。


In [ ]:
import signal
import sys
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# =================配置区域=================
CONFIG = {
    'total_epochs': 100,
    'save_every_n_steps': 500,      # 每隔多少步保存一次备份
    'patience': 5,                  # 早停耐心值 (epoch数)
    'min_delta': 1e-4,              # 最小改进阈值
    'checkpoint_dir': './checkpoints',
    'best_model_path': './checkpoints/best_model.pt',
    'interrupt_path': './checkpoints/interrupt_checkpoint.pt',
}

# 确保目录存在
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

# =================全局状态=================
stop_requested = False
best_val_loss = float('inf')
epochs_without_improvement = 0

# =================信号处理=================
def signal_handler(sig, frame):
    global stop_requested
    print(f"\n[信号] 接收到信号 {sig} (SIGINT/SIGTERM). 准备优雅退出...")
    stop_requested = True

# 注册信号
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# =================辅助函数=================
def save_checkpoint(state, filename):
    """通用保存函数"""
    try:
        torch.save(state, filename)
        print(f"[保存] 模型已保存至: {filename}")
    except Exception as e:
        print(f"[错误] 保存失败: {e}")

def train_epoch(model, dataloader, optimizer, criterion, device):
    """执行一个 Epoch 的训练"""
    model.train()
    total_loss = 0
    steps = 0
    
    for batch_idx, batch in enumerate(dataloader):
        # 1. 检查中断信号 (在每个 batch 前检查，响应更快)
        if stop_requested:
            return total_loss / (steps + 1), batch_idx * len(batch[0]) # 返回平均loss和已处理样本数
        
        # --- 模拟训练步骤 ---
        # inputs, labels = batch[0].to(device), batch[1].to(device)
        # optimizer.zero_grad()
        # outputs = model(inputs)
        # loss = criterion(outputs, labels)
        # loss.backward()
        # optimizer.step()
        # ------------------
        
        # 模拟损失计算 (替换为真实逻辑)
        loss = torch.tensor(1.0 / (batch_idx + 1), requires_grad=True) 
        
        total_loss += loss.item()
        steps += 1
        
        # 2. 定期保存逻辑 (Step-level Checkpoint)
        global_step = (epoch * len(dataloader)) + batch_idx
        if (global_step > 0) and (global_step % CONFIG['save_every_n_steps'] == 0):
            save_checkpoint({
                'type': 'backup',
                'epoch': epoch,
                'step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss.item(),
            }, f"{CONFIG['checkpoint_dir']}/backup_step_{global_step}.pt")

    return total_loss / max(steps, 1), steps * len(batch[0])

def validate(model, val_dataloader, criterion, device):
    """执行验证"""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            # inputs, labels = batch[0].to(device), batch[1].to(device)
            # outputs = model(inputs)
            # loss = criterion(outputs, labels)
            
            # 模拟验证损失
            loss = torch.tensor(0.8 / (1 + epoch * 0.1)) 
            
            total_loss += loss.item()
    return total_loss / len(val_dataloader)

# =================主训练循环=================
def main():
    global stop_requested, best_val_loss, epochs_without_improvement, epoch
    
    # 初始化占位符 (实际使用时请替换为真实对象)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = nn.Linear(10, 2).to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # 占位符 DataLoader (实际使用时替换)
    train_loader = DataLoader(range(100), batch_size=10) 
    val_loader = DataLoader(range(20), batch_size=10)

    # [可选] 恢复训练逻辑
    start_epoch = 0
    if os.path.exists(CONFIG['interrupt_path']):
        print(f"[恢复] 发现中断检查点: {CONFIG['interrupt_path']}")
        checkpoint = torch.load(CONFIG['interrupt_path'])
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        epochs_without_improvement = checkpoint.get('patience_count', 0)
        print(f"[恢复] 从 Epoch {start_epoch} 继续，当前最佳 Val Loss: {best_val_loss}")

    print(f"[开始] 训练开始 (从 Epoch {start_epoch} 到 {CONFIG['total_epochs']})")

    for epoch in range(start_epoch, CONFIG['total_epochs']):
        start_time = time.time()
        
        # 1. 训练
        train_loss, samples_processed = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # 2. 检查是否在训练过程中被中断
        if stop_requested:
            print("\n[中断] 在训练过程中检测到停止信号，正在保存中断断点...")
            save_checkpoint({
                'type': 'interrupt',
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'best_val_loss': best_val_loss,
                'patience_count': epochs_without_improvement,
                'message': 'Interrupted during training'
            }, CONFIG['interrupt_path'])
            sys.exit(0)

        # 3. 验证
        val_loss = validate(model, val_loader, criterion, device)
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1}/{CONFIG['total_epochs']} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s")

        # 4. Early Stopping 逻辑
        if val_loss < best_val_loss - CONFIG['min_delta']:
            print(f"[改进] Val Loss 从 {best_val_loss:.4f} 下降到 {val_loss:.4f}. 保存最佳模型...")
            best_val_loss = val_loss
            epochs_without_improvement = 0
            
            # 保存最佳模型
            save_checkpoint({
                'type': 'best',
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'config': CONFIG
            }, CONFIG['best_model_path'])
        else:
            epochs_without_improvement += 1
            print(f"[无改进] 连续 {epochs_without_improvement}/{CONFIG['patience']} 个 Epoch 未提升.")

        if epochs_without_improvement >= CONFIG['patience']:
            print(f"[早停] 触发 Early Stopping (连续 {CONFIG['patience']} 个 Epoch 未提升).")
            print("训练结束，最佳模型已保存。")
            break

        # 5. 每个 Epoch 结束时检查中断信号 (用于在验证后、下一轮开始前退出)
        if stop_requested:
            print("\n[中断] 在 Epoch 结束时检测到停止信号，正在保存中断断点...")
            save_checkpoint({
                'type': 'interrupt',
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'best_val_loss': best_val_loss,
                'patience_count': epochs_without_improvement,
                'message': 'Interrupted at epoch end'
            }, CONFIG['interrupt_path'])
            sys.exit(0)

    print("[完成] 所有训练周期结束。")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"[致命错误] 发生未捕获异常: {e}")
        # 发生异常时也尝试保存一下，防止完全丢失
        # save_checkpoint({...}, 'error_checkpoint.pt') 
        sys.exit(1)